# HistGradientBoosting on CIC IoT-DIAD 2024 Binary IDS

This notebook trains **HistGradientBoosting** on the frozen binary IDS benchmark package and evaluates the result on validation, test, and OOD attack splits.

The split and feature definitions come from the packaged preprocessing artifacts, so every model notebook runs against the same exact dataset contract.


In [ ]:
from __future__ import annotations

import gc
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pyarrow.parquet as pq
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

MODEL_NAME = "hist_gb"
MODEL_LABEL = "HistGradientBoosting"
SEED = 42
DATASET_DIR = Path("/kaggle/input/cic-iot-diad-2024-binary-ids")
OUTPUT_DIR = Path("/kaggle/working") / MODEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LABEL_MAP = {"Benign": 0, "Attack": 1}
MODEL_PARAMS = {
  "loss": "log_loss",
  "learning_rate": 0.05,
  "max_iter": 300,
  "max_leaf_nodes": 31,
  "early_stopping": true,
  "validation_fraction": 0.1,
  "random_state": 42,
  "verbose": 1
}

print("Dataset dir:", DATASET_DIR)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def load_feature_columns() -> list[str]:
    payload = load_json(DATASET_DIR / "feature_columns.json")
    return list(payload["feature_columns"])


def load_split_arrays(split_name: str, feature_columns: list[str]) -> tuple[np.ndarray, np.ndarray]:
    path = DATASET_DIR / f"{split_name}.parquet"
    table = pq.read_table(path, columns=feature_columns + ["derived_label_binary"])
    y = (table.column("derived_label_binary").to_pandas() == "Attack").to_numpy(dtype=np.int8)
    X = table.select(feature_columns).to_pandas().to_numpy(dtype=np.float32, copy=True)
    del table
    gc.collect()
    print(f"{split_name}: X={X.shape}, y={y.shape}")
    return X, y


def evaluate_split(model, split_name: str, feature_columns: list[str]) -> dict:
    x_split, y_split = load_split_arrays(split_name, feature_columns)
    start = time.perf_counter()
    y_pred = np.asarray(model.predict(x_split), dtype=np.int8)
    elapsed = time.perf_counter() - start

    tn, fp, fn, tp = confusion_matrix(y_split, y_pred, labels=[0, 1]).ravel()
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    specificity = tn / max(1, tn + fp)
    accuracy = (tp + tn) / max(1, len(y_split))
    f1 = (2 * precision * recall) / max(1e-12, precision + recall)
    fpr = fp / max(1, fp + tn)

    metrics = {
        "rows": int(len(y_split)),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "specificity": float(specificity),
        "fpr": float(fpr),
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        },
        "inference_seconds": float(elapsed),
        "rows_per_second": float(len(y_split) / max(elapsed, 1e-9)),
    }
    del x_split, y_split, y_pred
    gc.collect()
    return metrics


def build_model(model_name: str):
    if model_name == "logreg":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(**MODEL_PARAMS)),
            ]
        )
    if model_name == "random_forest":
        return RandomForestClassifier(**MODEL_PARAMS)
    if model_name == "hist_gb":
        return HistGradientBoostingClassifier(**MODEL_PARAMS)
    if model_name == "catboost":
        return CatBoostClassifier(**MODEL_PARAMS)
    if model_name == "mlp":
        return Pipeline(
            [
                ("scaler", StandardScaler()),
                ("clf", MLPClassifier(**MODEL_PARAMS)),
            ]
        )
    raise ValueError(f"Unsupported model: {model_name}")


feature_columns = load_feature_columns()
print("Feature count:", len(feature_columns))


In [ ]:
x_train, y_train = load_split_arrays("train", feature_columns)

model = build_model(MODEL_NAME)
train_start = time.perf_counter()
if MODEL_NAME == "catboost":
    x_val_fit, y_val_fit = load_split_arrays("val", feature_columns)
    class_counts = np.bincount(y_train, minlength=2)
    benign_weight = max(class_counts) / max(1, class_counts[0])
    attack_weight = max(class_counts) / max(1, class_counts[1])
    model.set_params(class_weights=[float(benign_weight), float(attack_weight)])
    model.fit(x_train, y_train, eval_set=(x_val_fit, y_val_fit), use_best_model=False)
    del x_val_fit, y_val_fit
    gc.collect()
else:
    model.fit(x_train, y_train)
train_seconds = time.perf_counter() - train_start
print("Train seconds:", round(train_seconds, 2))

del x_train, y_train
gc.collect()


In [ ]:
results = {
    "model_name": MODEL_NAME,
    "model_label": MODEL_LABEL,
    "seed": SEED,
    "train_seconds": float(train_seconds),
    "feature_count": len(feature_columns),
    "splits": {
        "val": evaluate_split(model, "val", feature_columns),
        "test": evaluate_split(model, "test", feature_columns),
        "ood_attack_holdout": evaluate_split(model, "ood_attack_holdout", feature_columns),
    },
}

metrics_summary = {
    "model": MODEL_NAME,
    "train_seconds": results["train_seconds"],
    "val_f1": results["splits"]["val"]["f1"],
    "test_f1": results["splits"]["test"]["f1"],
    "test_recall": results["splits"]["test"]["recall"],
    "test_precision": results["splits"]["test"]["precision"],
    "test_fpr": results["splits"]["test"]["fpr"],
    "ood_recall": results["splits"]["ood_attack_holdout"]["recall"],
}
print(json.dumps(metrics_summary, indent=2))

(OUTPUT_DIR / "results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
(OUTPUT_DIR / "metrics_summary.json").write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")
if MODEL_NAME == "catboost":
    model.save_model(OUTPUT_DIR / f"{MODEL_NAME}.cbm")
else:
    joblib.dump(model, OUTPUT_DIR / f"{MODEL_NAME}.joblib")
